# Merge Refinitiv & Trucost database
## Import libraries

In [40]:
import pandas as pd
import numpy as np
from unidecode import unidecode

## Functions

In [41]:
# Function to rename a string named 'FY0' to '2022', 'FY-1' to '2021', 'FY-2' to '2020', etc.
def rename_fiscal_year(fiscal_year):
    current_year = 2022
    if fiscal_year == 'FY0':
        return str(current_year)
    elif fiscal_year.startswith('FY-'):
        year_offset = int(fiscal_year[3:])
        renamed_year = current_year - year_offset
        return str(renamed_year)
    else:
        return fiscal_year
    

## Constants

In [42]:
comp_name_dict_erase = {
',':           '',         
'\.':          '',         
'corporation': '',
'corp':        '',       
'berhard':     '',    
'bhd':         '',        
'limited':     '',    
'ltd':         '',        
'plc':         '',        
'co':          '',         
'company':     ''}

## Refinitiv controls
### Import data

Put the first columns about the companies in this order
![Index columns order](assets/index_order.png)

In [43]:
# Importing data from Refinitiv.
# Has 2 levels of column names and 6 index to decribe a company.
refinitiv_controls = pd.read_excel('data/Refinitiv_ Controls.xlsx', sheet_name="All controls_Agri", header=[0,1], index_col=[0,1,2,3,4,5])
# Rename the index because they disappeared.
refinitiv_controls.index.names=['Identifier (RIC)','Company Name','Date Became Public','NAICS Industry Group Name','Business Description','Ticker Symbol']
refinitiv_controls.columns.names=['Variable','Fiscal Year']

### Clean data

In [44]:
# Remove SD column
refinitiv_controls.drop('Earnings Per Share - Standard Deviation\n(USD)\nIn the last 18 Y',axis=1,inplace=True, level=0)
# Rename the column names based on fiscal years
refinitiv_controls.rename(columns=rename_fiscal_year, inplace=True, level=1)
# Remove year 2022
refinitiv_controls.drop('2022',axis=1,inplace=True, level=1)


### Clean compagny names

In [45]:
# Reset index to column to be able to do modification to it
refinitiv_controls.reset_index(level="Company Name", inplace=True)

# Clean company names of the trucost dataset. Transform to ASCII, lower and erease characters
company_name_clean = refinitiv_controls['Company Name'].apply(unidecode).str.lower().replace(comp_name_dict_erase, regex=True)
        
# Insert new compagny name in new column
refinitiv_controls.insert(1, 'Company Name Clean', company_name_clean)

# Set two column as index
refinitiv_controls.set_index(keys=['Company Name','Company Name Clean'], append=True, inplace=True)

### Reshape data

In [46]:
refinitiv_controls_stacked = refinitiv_controls.stack(level=1, dropna=False)

## Refinitiv Beta
TODO

## Refinitiv returns
### Import data

It's necesserry to add year/month in the column header for returns per month
![Change in columns header](assets\returns_month.png)

In [47]:
refinitiv_returns = pd.read_excel('data/Refinitiv_ Returns.xlsx', sheet_name="Agriculture Comp, Full + Ticker", header=[0,1,2], index_col=[0,1,2,3,4,5])
# Rename index and columns
refinitiv_returns.index.names=['Identifier (RIC)','Company Name','Date Became Public','NAICS Industry Group Name','Business Description','Ticker Symbol']
refinitiv_returns.columns.names=['Variable','Fiscal Year', 'Fiscal Month']

### Clean data

In [48]:
# Remove return per year columns
refinitiv_returns_month = refinitiv_returns.drop('Total Return by Year (01.01.2004-31.12.2022)',axis=1,level=0)
# Remove year 2004, 2022
refinitiv_returns_month.drop([2004,2022],axis=1,inplace=True, level=1)

### Generate yearly returns statistics

In [49]:
# Gerenate column with available fiscal months per year
refinitiv_returns_count = refinitiv_returns_month.groupby(axis=1, level=1).count()
# Add header name
refinitiv_returns_count = pd.concat([refinitiv_returns_count], axis=1, keys=["Available fiscal months"])

In [50]:
# Generate mean for each year
refinitiv_returns_mean = refinitiv_returns_month.groupby(axis=1, level=1).mean()
# Add header name
refinitiv_returns_mean = pd.concat([refinitiv_returns_mean], axis=1, keys=["Return Mean"])


In [51]:
# Generate composed return for each year
refinitiv_returns_composed = (refinitiv_returns_month+1).groupby(axis=1, level=1).prod()-1
# Add header name
refinitiv_returns_composed = pd.concat([refinitiv_returns_composed], axis=1, keys=["Composed Return"])


In [52]:
# Generate Standard Deviation for each year
refinitiv_returns_std = refinitiv_returns_month.groupby(axis=1, level=1).std()
# Add header name
refinitiv_returns_std = pd.concat([refinitiv_returns_std], axis=1, keys=["Return Standard Deviation"])

In [53]:
# Put every result in the same dataframe
refinitiv_returns_year = pd.concat([refinitiv_returns_count, refinitiv_returns_mean, refinitiv_returns_composed, refinitiv_returns_std], axis=1)


### Reshape data

In [54]:
refinitiv_returns_year_stacked = refinitiv_returns_year.stack(level=1, dropna=False)

## Merge Refinitiv datas

In [55]:
refinitiv_data = refinitiv_controls_stacked.reset_index().join(refinitiv_returns_year_stacked.reset_index(drop=True)).set_index(refinitiv_controls_stacked.index.names)

## Export Refinitiv datas


In [56]:
# refinitiv_data.to_excel('data/Refinitiv_merged.xlsx')

## Trucost
### Import data
Put the first columns about the companies in this order
![Trucost Excel Index](assets/trucost_index_order.png)

In [68]:
trucost = pd.read_excel('data/Trucost 2004-2021 no duplicates.xlsx',  header=[0], index_col=[0,1,2,3,4,5,6,7,8])

### Clean data

### Clean compagny names

In [69]:
# Reset index to column to be able to do modification to it
trucost.reset_index(level="Company Name", inplace=True)

# Clean company names of the trucost dataset. Transform to ASCII, lower and erease characters
company_name_clean = trucost['Company Name'].apply(unidecode).str.lower().replace(comp_name_dict_erase, regex=True)
        
# Insert new compagny name in new column
trucost.insert(1, 'Company Name Clean', company_name_clean)

# Set two column as index
trucost.set_index(keys=['Company Name','Company Name Clean'], append=True, inplace=True)

In [70]:
trucost

Period End Date  \
Institution ID Ticker Status               Company Type    Country Year Founded Web Page               Fiscal Year Company Name                            Company Name Clean                                      
103333         ARGO   Operating            Public Company  Bermuda 1957         www.argolimited.com    2016        Argo Group International Holdings, Ltd. argo group international holdings          2016-12-31   
                                                                                                       2017        Argo Group International Holdings, Ltd. argo group international holdings          2017-12-31   
                                                                                                       2018        Argo Group International Holdings, Ltd. argo group international holdings          2018-12-31   
                                                                                                       2019        Argo Group International Holdings, Ltd. argo group international holdings          2019-12-31   
                                                                                                       2020        Argo Group International Holdings, Ltd. argo group international holdings          2020-12-31   
...                                                                                                                                                                                                          ...   
27762653       6601   Operating Subsidiary Public Company  China   2006         www.cheerwin.com       2020        Cheerwin Group Limited                  cheerwin group                             2020-12-31   
28714179       APPH   Operating            Public Company  Germany 1946         apontis-pharma.de      2020        Apontis Pharma AG                       apontis pharma ag                          2020-12-31   
                                                                                                       2021        Apontis Pharma AG                       apontis pharma ag                          2021-12-31   
29427292       BFG    Operating            Public Company  Sweden  1936         www.byggfaktagroup.com 2020        Byggfakta Group Nordic HoldCo AB (publ) byggfakta group nordic hold ab (publ)      2020-12-31   
29459940       AST    Operating            Private Company Romania 2017         arcticstream.ro        2021        Arctic Stream S.A.                      arctic stream sa                           2021-06-30   

                                                                                                                                                                                                  Intensity: GHG Scope 1  \
Institution ID Ticker Status               Company Type    Country Year Founded Web Page               Fiscal Year Company Name                            Company Name Clean                                              
103333         ARGO   Operating            Public Company  Bermuda 1957         www.argolimited.com    2016        Argo Group International Holdings, Ltd. argo group international holdings                    0.551230   
                                                                                                       2017        Argo Group International Holdings, Ltd. argo group international holdings                    0.539677   
                                                                                                       2018        Argo Group International Holdings, Ltd. argo group international holdings                    0.525665   
                                                                                                       2019        Argo Group International Holdings, Ltd. argo group international holdings                    0.508256   
                                                                                                       2020        A